# Station Information

* Station name: CUNSEY BECK_ESTHWAITE OUTFALL_E_202303
* Water Quality Data link: https://environment.data.gov.uk/hydrology/station/E04568A

## 0. Import the Relevant Libraries

In [1]:
# Force installation of Syne-Tune and a compatible NumPy version (may require restarting the session after running this cell before running below import cell)
!pip install numpy==1.26.1 syne_tune --force-reinstall -q # ignore potential error messages if listed dependencies do not match imports

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.3/254.3 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 136.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 153.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 141.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# Import relevant libraries (Note: May require running of above cell to ensure compatibility of NumPy and Syne-Tune)
import torch
import logging
logging.basicConfig(level=logging.INFO)
import numpy as np
import matplotlib.pyplot as plt
from syne_tune import StoppingCriterion, Tuner
from syne_tune.backend.python_backend.python_backend import PythonBackend
from syne_tune.config_space import loguniform, randint, choice, uniform
from syne_tune.experiments import load_experiment
from syne_tune.optimizer.baselines import ASHA
import pandas as pd
from pathlib import Path

##1. Specify Epoch Ranges and Reduction Factor for Hyperparameter Optimisation (HPO)

In [2]:
# Specify the minimum and maximum number of epochs for which a trial is allowed to run during HPO
min_number_of_epochs = 2
max_number_of_epochs = 10

# Define the reduction factor for the Asynchronous Successive Halving Algorithm (ASHA)
eta = 2 # Set to 2 so that 50% of trials are stopped at each rung of the scheduler

##2. Define the Tune Function to Optimise Hyperparameters for Ammonium Prediction

In [3]:
# Define the tune function for optimising LSTM hyperparameters using ASHA on ammonium_t_96 targets
def ammonium_hpo_objective_lstm_synetune(learning_rate, num_hiddens, num_layers, dropout, weight_decay):
    """
    Defines a function for optimising the hyperparameters of an LSTM model.

    Args:
      - learning_rate (float): The learning rate for the model.
      - num_hiddens (int): The number of hidden units per layer in the model.
      - num_layers (int): The number of hidden layers in the model.
      - dropout (float): The dropout regularisation rate for the model.
      - weight_decay (float): The weight decay regularisation rate for the model.

    Side effects:
      - Trains an LSTM for specified hyperparameter configuration and reports validation performance.
    """
    # Import reporter from Syne Tune
    from syne_tune import Reporter

    # Import base scripts within function for compatibility with Python Backend
    import pandas as pd
    import numpy as np
    import urllib.request
    import sys
    from pathlib import Path
    import io
    from sklearn.preprocessing import MinMaxScaler
    import torch

    # Import the Base Classes and Helper Functions from the Project GitHub
    url = "https://raw.githubusercontent.com/VinceMoran/EA_Water_Quality_Time_Series_Prediction/main/base_classes_and_helper_functions.py"
    file_path = Path("base_classes_and_helper_functions.py")
    if not file_path.exists():
        urllib.request.urlretrieve(url, file_path)
    sys.path.append(str(file_path.parent))
    import base_classes_and_helper_functions as bchf

    # Set the random seed for all PNRGs to ensure reproducibility
    bchf.set_random_seed()

    ### NOTE: DATA LOADING AND PREPROCESSING MUST BE DEFINED WITHIN THE SAME FUNCTION AS MODELLING CODE FOR COMPATIBILITY WITH SYNE-TUNE BACKEND!!!!!
    # Create the path to the directory containing the data in the Jupyter notebook environment
    data_path = bchf.load_raw_data(url="https://github.com/VinceMoran/EA_Water_Quality_Time_Series_Prediction/raw/main/data/processed_data/windermere_inflows_and_outflow/cunsey_beck/CUNSEY%20BECK_ESTHWAITE%20OUTFALL_E_202303/CUNSEY%20BECK_ESTHWAITE%20OUTFALL_E_202303_preprocessed_data.zip")
    # Assign the full filepath for the preprocessed data
    parameter_path = "/content" / data_path / "CUNSEY BECK_ESTHWAITE OUTFALL_E_202303_preprocessed_data.csv"
    df_parameters = pd.read_csv(parameter_path)
    # Change the data type for the dateTime column to datetime64
    df_parameters["dateTime"] = pd.to_datetime(df_parameters["dateTime"], dayfirst=False)
    # Set the datetime column as the index
    df_parameters.set_index('dateTime', inplace=True)
    # Numerically encode date and time as features
    df_parameters["day_of_year"] = df_parameters.index.dayofyear
    df_parameters["day_of_year_sin"] = np.sin(2*np.pi*(df_parameters["day_of_year"]/365))
    df_parameters["day_of_year_cos"] = np.cos(2*np.pi*(df_parameters["day_of_year"]/365))
    df_parameters["minute_of_day"] = df_parameters.index.hour*60 + df_parameters.index.minute
    df_parameters["minute_of_day_sin"] = np.sin(2*np.pi*(df_parameters["minute_of_day"]/1440))
    df_parameters["minute_of_day_cos"] = np.cos(2*np.pi*(df_parameters["minute_of_day"]/1440))
    # Remove redundant columns
    df_parameters.drop(["day_of_year", "minute_of_day"], axis=1, inplace=True)
    # Create lagged variables for each feature
    variables = ["ammonium", "conductivity", "oxygen_conc", "oxygen_perc", "temperature", "turbidity"]
    for i in np.arange(1, 5):
        for variable in variables:
            df_parameters[f"{variable}_lag{i}"] = df_parameters[variable].shift(periods=i)
    # Create moving averages for each water quality variable
    moving_average_windows = np.array([6, 12, 24]) # hours
    timestep_windows = moving_average_windows * 4 # number of timesteps
    # Loop through creating moving averages from only past values to avoid temporal leakage
    for i in timestep_windows:
        for variable in variables:
            df_parameters[f"{variable}_ma{i}"] = df_parameters[variable].rolling(window=i, min_periods=1).mean()
    # Create seasonal lagged features
    target_vars = ["ammonium", "oxygen_conc", "temperature"]
    seasonality = 96 # 24 hours prior in 15 minute sensor data
    for target in target_vars:
        df_parameters[f"{target}_seasonal_lag{seasonality}"] = df_parameters[target].shift(periods=seasonality)
    # Loop through creating moving averages of imputed value counts from only past values to avoid temporal leakage
    for i in timestep_windows:
        for variable in variables:
            df_parameters[f"{variable}_imputed_count_{i}"] = (df_parameters[f"{variable}_impossible"] + df_parameters[f"{variable}_anomaly"]).rolling(window=i, min_periods=1).sum()
    # Create the target variable for HPO
    target_var = "ammonium"
    target_timestep = 96
    df_parameters[f"{target_var}_target_t_{str(target_timestep)}"] = df_parameters[target_var].shift(periods=-target_timestep)
    # Remove NaN rows added after shifting features and targets
    df_parameters.dropna(inplace=True)
    # Split the DataFrame into training, validation, and testing sets
    df_train, df_val, df_test = bchf.train_val_test_split(df_parameters,
                                                          train_size=0.6,
                                                          val_size=0.2,
                                                          test_size=0.2)
    # Initialise the scaler using min-max normalisation (avoid temporal leakage by splitting into training, validation, and testing before fitting the scaler)
    feature_scaler = MinMaxScaler()
    # Create copies of the DataFrames for scaling
    df_train_scaled, df_val_scaled, df_test_scaled = df_train.copy(), df_val.copy(), df_test.copy()
    # Create a list of features for scaling
    features_to_scale = df_train.columns
    # Drop anomaly and impossible flags from the features to be scaled list using loop
    var_sub_names = ["ammonium", "conductivity", "oxygen_conc", "oxygen_perc", "temperature", "turbidity"]
    for var in var_sub_names:
        features_to_scale = features_to_scale.drop(f"{var}_impossible")
        features_to_scale = features_to_scale.drop(f"{var}_anomaly")
    # Drop temporal features from the list of features to be scaled
    features_to_scale = features_to_scale.drop("day_of_year_sin")
    features_to_scale = features_to_scale.drop("day_of_year_cos")
    features_to_scale = features_to_scale.drop("minute_of_day_sin")
    features_to_scale = features_to_scale.drop("minute_of_day_cos")
    # Remove target from the list of features for scaling
    features_to_scale = features_to_scale.drop(f"{target_var}_target_t_{target_timestep}")
    # Fit the scaler to the features during training
    feature_scaler.fit(df_train_scaled[features_to_scale])
    # Transform the features using min-max normalisation
    df_train_scaled[features_to_scale] = feature_scaler.transform(df_train_scaled[features_to_scale])
    df_val_scaled[features_to_scale] = feature_scaler.transform(df_val_scaled[features_to_scale])
    df_test_scaled[features_to_scale] = feature_scaler.transform(df_test_scaled[features_to_scale])
    # Change data types to float32 and int8 to reduce memory and speed up training
    float_cols = df_train_scaled.select_dtypes(include='float64').columns # identify column names containing 64 bit floating point data
    bool_cols = df_train_scaled.select_dtypes(include='bool').columns # identify column names containing Boolean data
    # Loop through DataFrames to convert columns to appropriate data types
    for df in [df_train_scaled, df_val_scaled, df_test_scaled]:
        df[float_cols] = df[float_cols].astype('float32')
        df[bool_cols] = df[bool_cols].astype('int8')
    # Obtain a mapping to the HPO target
    hpo_target = f"{target_var}_target_t_{str(target_timestep)}"
    # Create list of features
    features = df_train_scaled.columns.drop(hpo_target)
    # Create array for training features
    X_train = df_train_scaled[features].values
    # Create array for training targets
    y_train = df_train_scaled[hpo_target].values
    # Create array for validation features
    X_val = df_val_scaled[features].values
    # Create array for validation targets
    y_val = df_val_scaled[hpo_target].values
    # Create array for testing features
    X_test = df_test_scaled[features].values
    # Create array for testing targets
    y_test = df_test_scaled[hpo_target].values
    # Instantiate the target scaler object
    target_scaler = MinMaxScaler()
    # Fit the scaler to the training targets
    target_scaler.fit(y_train.reshape(-1, 1))
    # Scale training targets
    y_train = target_scaler.transform(y_train.reshape(-1, 1))
    # Scale the validation targets
    y_val = target_scaler.transform(y_val.reshape(-1, 1))
    # Scale the testing targets
    y_test = target_scaler.transform(y_test.reshape(-1, 1))
    # Instantiate the TimeSeries data class for HPO
    data = bchf.TimeSeries(batch_size=32, # minibatch size defaults to 32 to balance convergence speed with gradient estimation accuracy
                           num_steps=96, # sliding window of 96 represents 24hrs of 15 minute resolution data
                           X_train=X_train,
                           y_train=y_train,
                           X_val=X_val,
                           y_val=y_val,
                           X_test=X_test,
                           y_test=y_test)

    ### NOTE: MODELLING CODE BEGINS HERE!!!!!
    # Specify the maximum number of epochs for which a trial is allowed to run during HPO
    MAX_NUMBER_OF_EPOCHS = 10

    # Obtain the shape of the input tensor to be passed into the LSTM
    num_inputs = next(iter(data.train_dataloader()))[0].shape[2]
    # Specify the model hyperparameters
    lstm = bchf.LSTM(num_inputs=num_inputs,
                     num_hiddens=num_hiddens,
                     num_layers=num_layers,
                     dropout=dropout)
    # Create an instance of the model class
    model = bchf.RNNWQ(rnn=lstm,
                       out_features=1,
                       lr=learning_rate,
                       weight_decay=weight_decay)
    # Create an instance of the HPOTrainer class
    trainer = bchf.HPOTrainer(max_epochs=1, num_gpus=1) # `max_epochs` set to 1 so that each call to `fit_epoch()` in the below loop runs for 1 epoch per iteration
    # Create the reporting object to send metrics to syne tune during training
    report = Reporter()
    # Loop through each epoch up until the specified number of maximum epochs
    for epoch in range(1, MAX_NUMBER_OF_EPOCHS + 1):
        # Initialise the trainer on the first epoch
        if epoch == 1:
            # Initialize the state of Trainer
            trainer.fit(model=model, data=data)
        # Fit for an additional epoch after the first epoch
        else:
            trainer.fit_epoch()
        # Compute the validation error
        validation_error = trainer.validation_error().cpu().detach().numpy()
        # Report the validation error from each epoch
        report(epoch=epoch, validation_error=float(validation_error))

## 3. Define the Configuration Space for Hyperparameter Search

In [4]:
# Create a dictionary defining the space of hyperparameters to explore
config_space = {
    "learning_rate": loguniform(1e-5, 5e-4), # explores learning rates sampled from a lognormal distribution between 0.00001 and 0.0005 (biases lower rates since higher learning rates were often unstable)
    "num_hiddens": choice([32, 64, 128, 256]), # explores a range of hidden units between 32 and 256 (multiples of 8 for computational efficiency)
    "num_layers": randint(2, 4), # explores 2 to 4 hidden layers (initial trials showed optimal configurations were usually in this range)
    "dropout": uniform(0.2, 0.5), # explores moderate to slightly aggressive dropout rates from a uniform distribution between 0.2 and 0.5
    "weight_decay": loguniform(1e-5, 5e-4), # explores weight decay coefficients sampled from a lognormal distribution between 0.00001 and 0.0005
}

# Define the initial configuration for the HPO searcher to try first (this was found to be a very strong baseline for most stations)
initial_config = {
    "learning_rate": 0.00001,
    "num_hiddens": 64,
    "num_layers": 2,
    "dropout": 0.2,
    "weight_decay": 0.00001
}

##4. Define the Resources Available for HPO

In [5]:
# Define the number of parallel workers available to evaluate trials concurrently
n_workers = max(1, torch.cuda.device_count()) # Uses all available GPUs for parallel trials or defaults to 1 worker if running on CPU

# Define the maximum number of trials to test
max_number_of_trials = 30

# Define the stopping criterion for HPO
stop_criterion = StoppingCriterion(max_num_trials_started=max_number_of_trials)

##5. Define Asynchronous Successive Halving Algorithm (ASHA) for HPO

In [6]:
# Define ASHA configurations for HPO
minimise_mode = True # ASHA seeks to minimise the metric if True
metric = "validation_error" # The metric to be optimised during HPO is validation error
resource_attr = "epoch" # ASHA should track optimisation over epochs and stop low-performing trials early

# Initialise the ASHA HPO scheduler
scheduler = ASHA(
    config_space, # search over the defined hyperparameter space
    metric=metric, # optimise the chosen metric (validation error)
    do_minimize=minimise_mode, # ASHA should minimise the metric
    points_to_evaluate=[initial_config], # seed the search with an initial baseline configuration
    max_t=max_number_of_epochs, # maximum number of epochs per trial
    time_attr=resource_attr # track trial progress by epochs
    )

##6. Initialise and Run the Tuner for Ammonium HPO

In [7]:
# Instruct the searcher to run each trial as a Python function
trial_backend = PythonBackend(
    tune_function=ammonium_hpo_objective_lstm_synetune, # use the predefined hpo_objective_lstm_synetune function to optimise hyperparameters
    config_space=config_space) # sample hyperparameters from the predefined configuration space

In [8]:
# Initialise the tuner object for HPO
tuner = Tuner(
    trial_backend=trial_backend, # pass the predefined backend function (`hpo_objective_lstm_synetune`) for running each hyperparameter trial
    scheduler=scheduler, # pass the predefined scheduler (ASHA)
    stop_criterion=stop_criterion, # pass the stopping criterion (`max_number_of_trials`)
    n_workers=n_workers, # pass the number of parallel workers available for use
    print_update_interval=int(max_number_of_trials * 0.5), # print updates after 50% of the maximum number of trials have been started
)

In [9]:
# Run the HPO experiment
tuner.run()


🚀 Syne Tune - Hyperparameter Optimization
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋 Experiment Configuration
├─ Name: python-entrypoint-2026-03-26-11-33-01-155
├─ Backend: PythonBackend
├─ Workers: 1
├─ Scheduler: ASHA
├─ Results Path: /root/syne-tune/python-entrypoint-2026-03-26-11-33-01-155
└─ Log Path: /root/syne-tune/python-entrypoint-2026-03-26-11-33-01-155

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🏁 Starting hyperparameter optimization...
[11:33:02] 🚀 Trial 0 started - config: learning_rate=1e-05, num_hiddens=64, num_layers=2, dropout=0.2, weight_decay=1e-05

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 Tuning Status (last metric is reported)
 trial_id     status  iter  learning_rate  num_hiddens  num_layers  dropout  weight_decay
        0 InProgress     0        0.00001           64           2      0.2       0.00001
1 trials running, 0 finished (0 until the

##7. Load Results from HPO Experiment to a Pandas DataFrame

In [10]:
# Assign the filepath for the results of the HPO experiment
ammonium_results_path = "/root/syne-tune/python-entrypoint-2026-03-26-11-33-01-155/results.csv.zip"

# Save the results to a Pandas DataFrame
ammonium_results = pd.read_csv(ammonium_results_path)

##8. Save the Ammonium HPO Results as a .csv File

In [11]:
# Specify the file path for saving the DataFrame as a csv file
ammonium_hpo_results_path = Path("/content/CUNSEY BECK_ESTHWAITE OUTFALL_E_202303_ammonium_hpo_results.csv")

# Save the DataFrame to a csv file
ammonium_results.to_csv(ammonium_hpo_results_path, index=True)

## 9. Define the Config Space, Scheduler, and Tune Function for Dissolved Oxygen HPO

In [12]:
# Create a dictionary defining the space of hyperparameters to explore
config_space = {
    "learning_rate": loguniform(1e-5, 5e-4), # explores learning rates sampled from a lognormal distribution between 0.00001 and 0.0005 (biases lower rates since higher learning rates were often unstable)
    "num_hiddens": choice([32, 64, 128, 256]), # explores a range of hidden units between 32 and 256 (multiples of 8 for computational efficiency)
    "num_layers": randint(2, 4), # explores 2 to 4 hidden layers (initial trials showed optimal configurations were usually in this range)
    "dropout": uniform(0.2, 0.5), # explores moderate to slightly aggressive dropout rates from a uniform distribution between 0.2 and 0.5
    "weight_decay": loguniform(1e-5, 5e-4), # explores weight decay coefficients sampled from a lognormal distribution between 0.00001 and 0.0005
}

# Define the initial configuration for the HPO searcher to try first (this was found to be a very strong baseline for most stations)
initial_config = {
    "learning_rate": 0.00001,
    "num_hiddens": 64,
    "num_layers": 2,
    "dropout": 0.2,
    "weight_decay": 0.00001
}

In [13]:
# Define ASHA configurations for HPO
minimise_mode = True # ASHA seeks to minimise the metric if True
metric = "validation_error" # The metric to be optimised during HPO is validation error
resource_attr = "epoch" # ASHA should track optimisation over epochs and stop low-performing trials early

# Initialise the ASHA HPO scheduler
scheduler = ASHA(
    config_space, # search over the defined hyperparameter space
    metric=metric, # optimise the chosen metric (validation error)
    do_minimize=minimise_mode, # ASHA should minimise the metric
    points_to_evaluate=[initial_config], # seed the search with an initial baseline configuration
    max_t=max_number_of_epochs, # maximum number of epochs per trial
    time_attr=resource_attr # track trial progress by epochs
    )

In [15]:
# Define the tune function for optimising LSTM hyperparameters using ASHA on ammonium_t_96 targets
def oxygen_hpo_objective_lstm_synetune(learning_rate, num_hiddens, num_layers, dropout, weight_decay):
    """
    Defines a function for optimising the hyperparameters of an LSTM model.

    Args:
      - learning_rate (float): The learning rate for the model.
      - num_hiddens (int): The number of hidden units per layer in the model.
      - num_layers (int): The number of hidden layers in the model.
      - dropout (float): The dropout regularisation rate for the model.
      - weight_decay (float): The weight decay regularisation rate for the model.

    Side effects:
      - Trains an LSTM for specified hyperparameter configuration and reports validation performance.
    """
    # Import reporter from Syne Tune
    from syne_tune import Reporter

    # Import base scripts within function for compatibility with Python Backend
    import pandas as pd
    import numpy as np
    import urllib.request
    import sys
    from pathlib import Path
    import io
    from sklearn.preprocessing import MinMaxScaler
    import torch

    # Import the Base Classes and Helper Functions from the Project GitHub
    url = "https://raw.githubusercontent.com/VinceMoran/EA_Water_Quality_Time_Series_Prediction/main/base_classes_and_helper_functions.py"
    file_path = Path("base_classes_and_helper_functions.py")
    if not file_path.exists():
        urllib.request.urlretrieve(url, file_path)
    sys.path.append(str(file_path.parent))
    import base_classes_and_helper_functions as bchf

    # Set the random seed for all PNRGs to ensure reproducibility
    bchf.set_random_seed()

    ### NOTE: DATA LOADING AND PREPROCESSING MUST BE DEFINED WITHIN THE SAME FUNCTION AS MODELLING CODE FOR COMPATIBILITY WITH SYNE-TUNE BACKEND!!!!!
    # Create the path to the directory containing the data in the Jupyter notebook environment
    data_path = bchf.load_raw_data(url="https://github.com/VinceMoran/EA_Water_Quality_Time_Series_Prediction/raw/main/data/processed_data/windermere_inflows_and_outflow/cunsey_beck/CUNSEY%20BECK_ESTHWAITE%20OUTFALL_E_202303/CUNSEY%20BECK_ESTHWAITE%20OUTFALL_E_202303_preprocessed_data.zip")
    # Assign the full filepath for the preprocessed data
    parameter_path = "/content" / data_path / "CUNSEY BECK_ESTHWAITE OUTFALL_E_202303_preprocessed_data.csv"
    df_parameters = pd.read_csv(parameter_path)
    # Change the data type for the dateTime column to datetime64
    df_parameters["dateTime"] = pd.to_datetime(df_parameters["dateTime"], dayfirst=False)
    # Set the datetime column as the index
    df_parameters.set_index('dateTime', inplace=True)
    # Numerically encode date and time as features
    df_parameters["day_of_year"] = df_parameters.index.dayofyear
    df_parameters["day_of_year_sin"] = np.sin(2*np.pi*(df_parameters["day_of_year"]/365))
    df_parameters["day_of_year_cos"] = np.cos(2*np.pi*(df_parameters["day_of_year"]/365))
    df_parameters["minute_of_day"] = df_parameters.index.hour*60 + df_parameters.index.minute
    df_parameters["minute_of_day_sin"] = np.sin(2*np.pi*(df_parameters["minute_of_day"]/1440))
    df_parameters["minute_of_day_cos"] = np.cos(2*np.pi*(df_parameters["minute_of_day"]/1440))
    # Remove redundant columns
    df_parameters.drop(["day_of_year", "minute_of_day"], axis=1, inplace=True)
    # Create lagged variables for each feature
    variables = ["ammonium", "conductivity", "oxygen_conc", "oxygen_perc", "temperature", "turbidity"]
    for i in np.arange(1, 5):
        for variable in variables:
            df_parameters[f"{variable}_lag{i}"] = df_parameters[variable].shift(periods=i)
    # Create moving averages for each water quality variable
    moving_average_windows = np.array([6, 12, 24]) # hours
    timestep_windows = moving_average_windows * 4 # number of timesteps
    # Loop through creating moving averages from only past values to avoid temporal leakage
    for i in timestep_windows:
        for variable in variables:
            df_parameters[f"{variable}_ma{i}"] = df_parameters[variable].rolling(window=i, min_periods=1).mean()
    # Create seasonal lagged features
    target_vars = ["ammonium", "oxygen_conc", "temperature"]
    seasonality = 96 # 24 hours prior in 15 minute sensor data
    for target in target_vars:
        df_parameters[f"{target}_seasonal_lag{seasonality}"] = df_parameters[target].shift(periods=seasonality)
    # Loop through creating moving averages of imputed value counts from only past values to avoid temporal leakage
    for i in timestep_windows:
        for variable in variables:
            df_parameters[f"{variable}_imputed_count_{i}"] = (df_parameters[f"{variable}_impossible"] + df_parameters[f"{variable}_anomaly"]).rolling(window=i, min_periods=1).sum()
    # Create the target variable for HPO
    target_var = "oxygen_conc"
    target_timestep = 96
    df_parameters[f"{target_var}_target_t_{str(target_timestep)}"] = df_parameters[target_var].shift(periods=-target_timestep)
    # Remove NaN rows added after shifting features and targets
    df_parameters.dropna(inplace=True)
    # Split the DataFrame into training, validation, and testing sets
    df_train, df_val, df_test = bchf.train_val_test_split(df_parameters,
                                                          train_size=0.6,
                                                          val_size=0.2,
                                                          test_size=0.2)
    # Initialise the scaler using min-max normalisation (avoid temporal leakage by splitting into training, validation, and testing before fitting the scaler)
    feature_scaler = MinMaxScaler()
    # Create copies of the DataFrames for scaling
    df_train_scaled, df_val_scaled, df_test_scaled = df_train.copy(), df_val.copy(), df_test.copy()
    # Create a list of features for scaling
    features_to_scale = df_train.columns
    # Drop anomaly and impossible flags from the features to be scaled list using loop
    var_sub_names = ["ammonium", "conductivity", "oxygen_conc", "oxygen_perc", "temperature", "turbidity"]
    for var in var_sub_names:
        features_to_scale = features_to_scale.drop(f"{var}_impossible")
        features_to_scale = features_to_scale.drop(f"{var}_anomaly")
    # Drop temporal features from the list of features to be scaled
    features_to_scale = features_to_scale.drop("day_of_year_sin")
    features_to_scale = features_to_scale.drop("day_of_year_cos")
    features_to_scale = features_to_scale.drop("minute_of_day_sin")
    features_to_scale = features_to_scale.drop("minute_of_day_cos")
    # Remove target from the list of features for scaling
    features_to_scale = features_to_scale.drop(f"{target_var}_target_t_{target_timestep}")
    # Fit the scaler to the features during training
    feature_scaler.fit(df_train_scaled[features_to_scale])
    # Transform the features using min-max normalisation
    df_train_scaled[features_to_scale] = feature_scaler.transform(df_train_scaled[features_to_scale])
    df_val_scaled[features_to_scale] = feature_scaler.transform(df_val_scaled[features_to_scale])
    df_test_scaled[features_to_scale] = feature_scaler.transform(df_test_scaled[features_to_scale])
    # Change data types to float32 and int8 to reduce memory and speed up training
    float_cols = df_train_scaled.select_dtypes(include='float64').columns # identify column names containing 64 bit floating point data
    bool_cols = df_train_scaled.select_dtypes(include='bool').columns # identify column names containing Boolean data
    # Loop through DataFrames to convert columns to appropriate data types
    for df in [df_train_scaled, df_val_scaled, df_test_scaled]:
        df[float_cols] = df[float_cols].astype('float32')
        df[bool_cols] = df[bool_cols].astype('int8')
    # Obtain a mapping to the HPO target
    hpo_target = f"{target_var}_target_t_{str(target_timestep)}"
    # Create list of features
    features = df_train_scaled.columns.drop(hpo_target)
    # Create array for training features
    X_train = df_train_scaled[features].values
    # Create array for training targets
    y_train = df_train_scaled[hpo_target].values
    # Create array for validation features
    X_val = df_val_scaled[features].values
    # Create array for validation targets
    y_val = df_val_scaled[hpo_target].values
    # Create array for testing features
    X_test = df_test_scaled[features].values
    # Create array for testing targets
    y_test = df_test_scaled[hpo_target].values
    # Instantiate the target scaler object
    target_scaler = MinMaxScaler()
    # Fit the scaler to the training targets
    target_scaler.fit(y_train.reshape(-1, 1))
    # Scale training targets
    y_train = target_scaler.transform(y_train.reshape(-1, 1))
    # Scale the validation targets
    y_val = target_scaler.transform(y_val.reshape(-1, 1))
    # Scale the testing targets
    y_test = target_scaler.transform(y_test.reshape(-1, 1))
    # Instantiate the TimeSeries data class for HPO
    data = bchf.TimeSeries(batch_size=32, # minibatch size defaults to 32 to balance convergence speed with gradient estimation accuracy
                           num_steps=96, # sliding window of 96 represents 24hrs of 15 minute resolution data
                           X_train=X_train,
                           y_train=y_train,
                           X_val=X_val,
                           y_val=y_val,
                           X_test=X_test,
                           y_test=y_test)

    ### NOTE: MODELLING CODE BEGINS HERE!!!!!
    # Specify the maximum number of epochs for which a trial is allowed to run during HPO
    MAX_NUMBER_OF_EPOCHS = 10

    # Obtain the shape of the input tensor to be passed into the LSTM
    num_inputs = next(iter(data.train_dataloader()))[0].shape[2]
    # Specify the model hyperparameters
    lstm = bchf.LSTM(num_inputs=num_inputs,
                     num_hiddens=num_hiddens,
                     num_layers=num_layers,
                     dropout=dropout)
    # Create an instance of the model class
    model = bchf.RNNWQ(rnn=lstm,
                       out_features=1,
                       lr=learning_rate,
                       weight_decay=weight_decay)
    # Create an instance of the HPOTrainer class
    trainer = bchf.HPOTrainer(max_epochs=1, num_gpus=1) # `max_epochs` set to 1 so that each call to `fit_epoch()` in the below loop runs for 1 epoch per iteration
    # Create the reporting object to send metrics to syne tune during training
    report = Reporter()
    # Loop through each epoch up until the specified number of maximum epochs
    for epoch in range(1, MAX_NUMBER_OF_EPOCHS + 1):
        # Initialise the trainer on the first epoch
        if epoch == 1:
            # Initialize the state of Trainer
            trainer.fit(model=model, data=data)
        # Fit for an additional epoch after the first epoch
        else:
            trainer.fit_epoch()
        # Compute the validation error
        validation_error = trainer.validation_error().cpu().detach().numpy()
        # Report the validation error from each epoch
        report(epoch=epoch, validation_error=float(validation_error))

##10. Initialise and Run the Tuner for Oxygen HPO

In [16]:
# Instruct the searcher to run each trial as a Python function
oxygen_trial_backend = PythonBackend(
    tune_function=oxygen_hpo_objective_lstm_synetune, # use the predefined hpo_objective_lstm_synetune function to optimise hyperparameters
    config_space=config_space) # sample hyperparameters from the predefined configuration space

In [17]:
# Initialise the tuner object for HPO
tuner = Tuner(
    trial_backend=oxygen_trial_backend, # pass the predefined backend function (`hpo_objective_lstm_synetune`) for running each hyperparameter trial
    scheduler=scheduler, # pass the predefined scheduler (ASHA)
    stop_criterion=stop_criterion, # pass the stopping criterion (`max_number_of_trials`)
    n_workers=n_workers, # pass the number of parallel workers available for use
    print_update_interval=int(max_number_of_trials * 0.5), # print updates after 50% of the maximum number of trials have been started
)

In [18]:
# Run the HPO experiment
tuner.run()


🚀 Syne Tune - Hyperparameter Optimization
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋 Experiment Configuration
├─ Name: python-entrypoint-2026-03-26-11-52-27-383
├─ Backend: PythonBackend
├─ Workers: 1
├─ Scheduler: ASHA
├─ Results Path: /root/syne-tune/python-entrypoint-2026-03-26-11-52-27-383
└─ Log Path: /root/syne-tune/python-entrypoint-2026-03-26-11-52-27-383

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🏁 Starting hyperparameter optimization...
[11:52:27] 🚀 Trial 0 started - config: learning_rate=1e-05, num_hiddens=64, num_layers=2, dropout=0.2, weight_decay=1e-05

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 Tuning Status (last metric is reported)
 trial_id     status  iter  learning_rate  num_hiddens  num_layers  dropout  weight_decay  epoch  validation_error  worker-time
        0 InProgress     2        0.00001           64           2      0.2       0.00001    

[12:09:19] ❌ Trial 28 failed.
[12:09:19] 🚀 Trial 29 started - config: learning_rate=7.072434892911124e-05, num_hiddens=32, num_layers=2, dropout=0.4988190408356249, weight_decay=0.0001365894588633698

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 Tuning Status (last metric is reported)
 trial_id     status  iter  learning_rate  num_hiddens  num_layers  dropout  weight_decay  epoch  validation_error  worker-time
        0  Completed    10       0.000010           64           2 0.200000      0.000010   10.0          0.043288    38.522051
        1    Stopped     9       0.000036          256           4 0.408441      0.000469    9.0          0.049185   132.895161
        2  Completed    10       0.000085          128           2 0.383711      0.000020   10.0          0.059390    41.837225
        3    Stopped     2       0.000014           32           4 0.363136      0.000149    2.0          0.091652    10.508831
        4    Stopped     9       0.0

##11. Load Results from Dissolved Oxygen HPO Experiment to a Pandas DataFrame

In [19]:
# Assign the filepath for the results of the HPO experiment
oxygen_results_path = "/root/syne-tune/python-entrypoint-2026-03-26-11-52-27-383/results.csv.zip"

# Save the results to a Pandas DataFrame
oxygen_results = pd.read_csv(oxygen_results_path)

##12. Save the Oxygen HPO Results as a .csv File

In [20]:
# Specify the file path for saving the DataFrame as a csv file
oxygen_hpo_results_path = Path("/content/CUNSEY BECK_ESTHWAITE OUTFALL_E_202303_oxygen_hpo_results.csv")

# Save the DataFrame to a csv file
oxygen_results.to_csv(oxygen_hpo_results_path, index=True)

## 13. Define the Config Space, Scheduler, and Tune Function for Temperature HPO

In [21]:
# Create a dictionary defining the space of hyperparameters to explore
config_space = {
    "learning_rate": loguniform(1e-5, 5e-4), # explores learning rates sampled from a lognormal distribution between 0.00001 and 0.0005 (biases lower rates since higher learning rates were often unstable)
    "num_hiddens": choice([32, 64, 128, 256]), # explores a range of hidden units between 32 and 256 (multiples of 8 for computational efficiency)
    "num_layers": randint(2, 4), # explores 2 to 4 hidden layers (initial trials showed optimal configurations were usually in this range)
    "dropout": uniform(0.2, 0.5), # explores moderate to slightly aggressive dropout rates from a uniform distribution between 0.2 and 0.5
    "weight_decay": loguniform(1e-5, 5e-4), # explores weight decay coefficients sampled from a lognormal distribution between 0.00001 and 0.0005
}

# Define the initial configuration for the HPO searcher to try first (this was found to be a very strong baseline for most stations)
initial_config = {
    "learning_rate": 0.00001,
    "num_hiddens": 64,
    "num_layers": 2,
    "dropout": 0.2,
    "weight_decay": 0.00001
}

In [22]:
# Define ASHA configurations for HPO
minimise_mode = True # ASHA seeks to minimise the metric if True
metric = "validation_error" # The metric to be optimised during HPO is validation error
resource_attr = "epoch" # ASHA should track optimisation over epochs and stop low-performing trials early

# Initialise the ASHA HPO scheduler
scheduler = ASHA(
    config_space, # search over the defined hyperparameter space
    metric=metric, # optimise the chosen metric (validation error)
    do_minimize=minimise_mode, # ASHA should minimise the metric
    points_to_evaluate=[initial_config], # seed the search with an initial baseline configuration
    max_t=max_number_of_epochs, # maximum number of epochs per trial
    time_attr=resource_attr # track trial progress by epochs
    )

In [23]:
# Define the tune function for optimising LSTM hyperparameters using ASHA on temperature_t_96 targets
def temperature_hpo_objective_lstm_synetune(learning_rate, num_hiddens, num_layers, dropout, weight_decay):
    """
    Defines a function for optimising the hyperparameters of an LSTM model.

    Args:
      - learning_rate (float): The learning rate for the model.
      - num_hiddens (int): The number of hidden units per layer in the model.
      - num_layers (int): The number of hidden layers in the model.
      - dropout (float): The dropout regularisation rate for the model.
      - weight_decay (float): The weight decay regularisation rate for the model.

    Side effects:
      - Trains an LSTM for specified hyperparameter configuration and reports validation performance.
    """
    # Import reporter from Syne Tune
    from syne_tune import Reporter

    # Import base scripts within function for compatibility with Python Backend
    import pandas as pd
    import numpy as np
    import urllib.request
    import sys
    from pathlib import Path
    import io
    from sklearn.preprocessing import MinMaxScaler
    import torch

    # Import the Base Classes and Helper Functions from the Project GitHub
    url = "https://raw.githubusercontent.com/VinceMoran/EA_Water_Quality_Time_Series_Prediction/main/base_classes_and_helper_functions.py"
    file_path = Path("base_classes_and_helper_functions.py")
    if not file_path.exists():
        urllib.request.urlretrieve(url, file_path)
    sys.path.append(str(file_path.parent))
    import base_classes_and_helper_functions as bchf

    # Set the random seed for all PNRGs to ensure reproducibility
    bchf.set_random_seed()

    ### NOTE: DATA LOADING AND PREPROCESSING MUST BE DEFINED WITHIN THE SAME FUNCTION AS MODELLING CODE FOR COMPATIBILITY WITH SYNE-TUNE BACKEND!!!!!
    # Create the path to the directory containing the data in the Jupyter notebook environment
    data_path = bchf.load_raw_data(url="https://github.com/VinceMoran/EA_Water_Quality_Time_Series_Prediction/raw/main/data/processed_data/windermere_inflows_and_outflow/cunsey_beck/CUNSEY%20BECK_ESTHWAITE%20OUTFALL_E_202303/CUNSEY%20BECK_ESTHWAITE%20OUTFALL_E_202303_preprocessed_data.zip")
    # Assign the full filepath for the preprocessed data
    parameter_path = "/content" / data_path / "CUNSEY BECK_ESTHWAITE OUTFALL_E_202303_preprocessed_data.csv"
    df_parameters = pd.read_csv(parameter_path)
    # Change the data type for the dateTime column to datetime64
    df_parameters["dateTime"] = pd.to_datetime(df_parameters["dateTime"], dayfirst=False)
    # Set the datetime column as the index
    df_parameters.set_index('dateTime', inplace=True)
    # Numerically encode date and time as features
    df_parameters["day_of_year"] = df_parameters.index.dayofyear
    df_parameters["day_of_year_sin"] = np.sin(2*np.pi*(df_parameters["day_of_year"]/365))
    df_parameters["day_of_year_cos"] = np.cos(2*np.pi*(df_parameters["day_of_year"]/365))
    df_parameters["minute_of_day"] = df_parameters.index.hour*60 + df_parameters.index.minute
    df_parameters["minute_of_day_sin"] = np.sin(2*np.pi*(df_parameters["minute_of_day"]/1440))
    df_parameters["minute_of_day_cos"] = np.cos(2*np.pi*(df_parameters["minute_of_day"]/1440))
    # Remove redundant columns
    df_parameters.drop(["day_of_year", "minute_of_day"], axis=1, inplace=True)
    # Create lagged variables for each feature
    variables = ["ammonium", "conductivity", "oxygen_conc", "oxygen_perc", "temperature", "turbidity"]
    for i in np.arange(1, 5):
        for variable in variables:
            df_parameters[f"{variable}_lag{i}"] = df_parameters[variable].shift(periods=i)
    # Create moving averages for each water quality variable
    moving_average_windows = np.array([6, 12, 24]) # hours
    timestep_windows = moving_average_windows * 4 # number of timesteps
    # Loop through creating moving averages from only past values to avoid temporal leakage
    for i in timestep_windows:
        for variable in variables:
            df_parameters[f"{variable}_ma{i}"] = df_parameters[variable].rolling(window=i, min_periods=1).mean()
    # Create seasonal lagged features
    target_vars = ["ammonium", "oxygen_conc", "temperature"]
    seasonality = 96 # 24 hours prior in 15 minute sensor data
    for target in target_vars:
        df_parameters[f"{target}_seasonal_lag{seasonality}"] = df_parameters[target].shift(periods=seasonality)
    # Loop through creating moving averages of imputed value counts from only past values to avoid temporal leakage
    for i in timestep_windows:
        for variable in variables:
            df_parameters[f"{variable}_imputed_count_{i}"] = (df_parameters[f"{variable}_impossible"] + df_parameters[f"{variable}_anomaly"]).rolling(window=i, min_periods=1).sum()
    # Create the target variable for HPO
    target_var = "temperature"
    target_timestep = 96
    df_parameters[f"{target_var}_target_t_{str(target_timestep)}"] = df_parameters[target_var].shift(periods=-target_timestep)
    # Remove NaN rows added after shifting features and targets
    df_parameters.dropna(inplace=True)
    # Split the DataFrame into training, validation, and testing sets
    df_train, df_val, df_test = bchf.train_val_test_split(df_parameters,
                                                          train_size=0.6,
                                                          val_size=0.2,
                                                          test_size=0.2)
    # Initialise the scaler using min-max normalisation (avoid temporal leakage by splitting into training, validation, and testing before fitting the scaler)
    feature_scaler = MinMaxScaler()
    # Create copies of the DataFrames for scaling
    df_train_scaled, df_val_scaled, df_test_scaled = df_train.copy(), df_val.copy(), df_test.copy()
    # Create a list of features for scaling
    features_to_scale = df_train.columns
    # Drop anomaly and impossible flags from the features to be scaled list using loop
    var_sub_names = ["ammonium", "conductivity", "oxygen_conc", "oxygen_perc", "temperature", "turbidity"]
    for var in var_sub_names:
        features_to_scale = features_to_scale.drop(f"{var}_impossible")
        features_to_scale = features_to_scale.drop(f"{var}_anomaly")
    # Drop temporal features from the list of features to be scaled
    features_to_scale = features_to_scale.drop("day_of_year_sin")
    features_to_scale = features_to_scale.drop("day_of_year_cos")
    features_to_scale = features_to_scale.drop("minute_of_day_sin")
    features_to_scale = features_to_scale.drop("minute_of_day_cos")
    # Remove target from the list of features for scaling
    features_to_scale = features_to_scale.drop(f"{target_var}_target_t_{target_timestep}")
    # Fit the scaler to the features during training
    feature_scaler.fit(df_train_scaled[features_to_scale])
    # Transform the features using min-max normalisation
    df_train_scaled[features_to_scale] = feature_scaler.transform(df_train_scaled[features_to_scale])
    df_val_scaled[features_to_scale] = feature_scaler.transform(df_val_scaled[features_to_scale])
    df_test_scaled[features_to_scale] = feature_scaler.transform(df_test_scaled[features_to_scale])
    # Change data types to float32 and int8 to reduce memory and speed up training
    float_cols = df_train_scaled.select_dtypes(include='float64').columns # identify column names containing 64 bit floating point data
    bool_cols = df_train_scaled.select_dtypes(include='bool').columns # identify column names containing Boolean data
    # Loop through DataFrames to convert columns to appropriate data types
    for df in [df_train_scaled, df_val_scaled, df_test_scaled]:
        df[float_cols] = df[float_cols].astype('float32')
        df[bool_cols] = df[bool_cols].astype('int8')
    # Obtain a mapping to the HPO target
    hpo_target = f"{target_var}_target_t_{str(target_timestep)}"
    # Create list of features
    features = df_train_scaled.columns.drop(hpo_target)
    # Create array for training features
    X_train = df_train_scaled[features].values
    # Create array for training targets
    y_train = df_train_scaled[hpo_target].values
    # Create array for validation features
    X_val = df_val_scaled[features].values
    # Create array for validation targets
    y_val = df_val_scaled[hpo_target].values
    # Create array for testing features
    X_test = df_test_scaled[features].values
    # Create array for testing targets
    y_test = df_test_scaled[hpo_target].values
    # Instantiate the target scaler object
    target_scaler = MinMaxScaler()
    # Fit the scaler to the training targets
    target_scaler.fit(y_train.reshape(-1, 1))
    # Scale training targets
    y_train = target_scaler.transform(y_train.reshape(-1, 1))
    # Scale the validation targets
    y_val = target_scaler.transform(y_val.reshape(-1, 1))
    # Scale the testing targets
    y_test = target_scaler.transform(y_test.reshape(-1, 1))
    # Instantiate the TimeSeries data class for HPO
    data = bchf.TimeSeries(batch_size=32, # minibatch size defaults to 32 to balance convergence speed with gradient estimation accuracy
                           num_steps=96, # sliding window of 96 represents 24hrs of 15 minute resolution data
                           X_train=X_train,
                           y_train=y_train,
                           X_val=X_val,
                           y_val=y_val,
                           X_test=X_test,
                           y_test=y_test)

    ### NOTE: MODELLING CODE BEGINS HERE!!!!!
    # Specify the maximum number of epochs for which a trial is allowed to run during HPO
    MAX_NUMBER_OF_EPOCHS = 10

    # Obtain the shape of the input tensor to be passed into the LSTM
    num_inputs = next(iter(data.train_dataloader()))[0].shape[2]
    # Specify the model hyperparameters
    lstm = bchf.LSTM(num_inputs=num_inputs,
                     num_hiddens=num_hiddens,
                     num_layers=num_layers,
                     dropout=dropout)
    # Create an instance of the model class
    model = bchf.RNNWQ(rnn=lstm,
                       out_features=1,
                       lr=learning_rate,
                       weight_decay=weight_decay)
    # Create an instance of the HPOTrainer class
    trainer = bchf.HPOTrainer(max_epochs=1, num_gpus=1) # `max_epochs` set to 1 so that each call to `fit_epoch()` in the below loop runs for 1 epoch per iteration
    # Create the reporting object to send metrics to syne tune during training
    report = Reporter()
    # Loop through each epoch up until the specified number of maximum epochs
    for epoch in range(1, MAX_NUMBER_OF_EPOCHS + 1):
        # Initialise the trainer on the first epoch
        if epoch == 1:
            # Initialize the state of Trainer
            trainer.fit(model=model, data=data)
        # Fit for an additional epoch after the first epoch
        else:
            trainer.fit_epoch()
        # Compute the validation error
        validation_error = trainer.validation_error().cpu().detach().numpy()
        # Report the validation error from each epoch
        report(epoch=epoch, validation_error=float(validation_error))

##14. Initialise and Run the Tuner for Temperature HPO

In [24]:
# Instruct the searcher to run each trial as a Python function
temperature_trial_backend = PythonBackend(
    tune_function=temperature_hpo_objective_lstm_synetune, # use the predefined hpo_objective_lstm_synetune function to optimise hyperparameters
    config_space=config_space) # sample hyperparameters from the predefined configuration space

In [25]:
# Initialise the tuner object for HPO
tuner = Tuner(
    trial_backend=temperature_trial_backend, # pass the predefined backend function (`hpo_objective_lstm_synetune`) for running each hyperparameter trial
    scheduler=scheduler, # pass the predefined scheduler (ASHA)
    stop_criterion=stop_criterion, # pass the stopping criterion (`max_number_of_trials`)
    n_workers=n_workers, # pass the number of parallel workers available for use
    print_update_interval=int(max_number_of_trials * 0.5), # print updates after 50% of the maximum number of trials have been started
)

In [26]:
# Run the HPO experiment
tuner.run()


🚀 Syne Tune - Hyperparameter Optimization
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋 Experiment Configuration
├─ Name: python-entrypoint-2026-03-26-12-10-18-406
├─ Backend: PythonBackend
├─ Workers: 1
├─ Scheduler: ASHA
├─ Results Path: /root/syne-tune/python-entrypoint-2026-03-26-12-10-18-406
└─ Log Path: /root/syne-tune/python-entrypoint-2026-03-26-12-10-18-406

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🏁 Starting hyperparameter optimization...
[12:10:19] 🚀 Trial 0 started - config: learning_rate=1e-05, num_hiddens=64, num_layers=2, dropout=0.2, weight_decay=1e-05

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 Tuning Status (last metric is reported)
 trial_id     status  iter  learning_rate  num_hiddens  num_layers  dropout  weight_decay  epoch  validation_error  worker-time
        0 InProgress     2        0.00001           64           2      0.2       0.00001    

##15. Load Results from Temperature HPO Experiment to a Pandas DataFrame

In [27]:
# Assign the filepath for the results of the HPO experiment
temperature_results_path = "/root/syne-tune/python-entrypoint-2026-03-26-12-10-18-406/results.csv.zip"

# Save the results to a Pandas DataFrame
temperature_results = pd.read_csv(temperature_results_path)

##16. Save the Temperature HPO Results as a .csv File

In [28]:
# Specify the file path for saving the DataFrame as a csv file
temperature_hpo_results_path = Path("/content/CUNSEY BECK_ESTHWAITE OUTFALL_E_202303_temperature_hpo_results.csv")

# Save the DataFrame to a csv file
temperature_results.to_csv(temperature_hpo_results_path, index=True)